In [ ]:
%pip install -q "edr-xarray[dask]"

# 03 - Backend options

Try discovery mode, caller-owned HTTP clients, and xarray-owned Dask chunking
with explicit collection and variable choices.

## 1. Set the server

In [ ]:
import httpx
import xarray as xr

import edr_xarray  # registers engine="edr"

server = "https://edr.example.com"

## 2. List collections

Run this cell, then choose one collection id in the next cell.

In [ ]:
response = httpx.get(f"{server}/collections", timeout=30.0)
response.raise_for_status()
collections = response.json()["collections"]

for collection in collections:
    print(collection["id"], "-", collection.get("title", ""))

## 3. Choose a collection

In [ ]:
collection_id = "example-collection"  # Change this to one of the ids listed above.
collection_url = f"{server}/collections/{collection_id}"
collection_url

## 4. Compare discovery modes

In [ ]:
metadata_ds = xr.open_dataset(collection_url, engine="edr", discovery="metadata_only")
print("metadata_only:", dict(metadata_ds.sizes))
metadata_ds.close()

probe_ds = xr.open_dataset(collection_url, engine="edr", discovery="probe")
print("probe:        ", dict(probe_ds.sizes))
probe_ds.close()

## 5. Open a specific instance

Some EDR collections expose forecast runs or versions as instances. Set `instance_id` to one advertised by the server. Opening with `instance=` reads collection metadata, then the selected instance metadata. With `discovery="metadata_only"`, this still does not fetch cube values.

In [ ]:
instance_id = ""  # Change this to an instance id such as "f024"; leave empty to skip.

if instance_id:
    instance_ds = xr.open_dataset(
        collection_url,
        engine="edr",
        instance=instance_id,
        discovery="metadata_only",
    )
    print("instance sizes:", dict(instance_ds.sizes))
    print("instance title:", instance_ds.attrs.get("title", ""))
    print("data variables:", list(instance_ds.data_vars))
    instance_ds.close()
else:
    print("Set instance_id to run this optional example.")

## 6. Use a caller-owned HTTP client

In [ ]:
client = httpx.Client(headers={"User-Agent": "edr-xarray-example"}, timeout=30.0)

ds = xr.open_dataset(collection_url, engine="edr", session=client)
ds.close()
client.close()

## 7. Open with Dask chunks

In [ ]:
import dask.array as da

ds = xr.open_dataset(collection_url, engine="edr", chunks={})

print("sizes:")
print(dict(ds.sizes))
print()
print("data variables:")
print(list(ds.data_vars))

## 8. Choose a variable and inspect Dask data

In [ ]:
variable = "temperature"  # Change this to one of the data variables listed above.

assert isinstance(ds[variable].data, da.Array)
ds[variable].data

## 9. Compute a small Dask result

Change `indexer` to dimensions shown in `ds.sizes`. `isel` uses
zero-based positions, so the largest valid integer position is `size - 1`.

In [ ]:
indexer = {"t": 0, "y": 0, "x": 0}
result = ds[variable].isel(indexer).compute()

result.values

## 10. Close resources

In [ ]:
ds.close()